# Gunter's Space Page — tabla tabular completa (una fila por objeto)

Este notebook junta **todo** lo ya descargado de Gunter's Space Page
(`data/gunter/`) en una sola tabla ancha, **una fila por objeto/lanzamiento
del registro `#satlist`** del sitio.

Objetivos:

1. **Sin filtrar ni clasificar**: solo ordena en columnas lo que el sitio ya
   dice. No hay clasificación de causas ni dataset derivado de fallas.
2. **Todas las variables disponibles**: campo `#satdescription` (narrativa en
   prosa), bloque `#satdata` (Nation, Operator, Mass, Orbit, …), registro de
   lanzamientos (COSPAR, fecha, sitio, vehículo, remarks) y años mencionados
   en la narrativa.
3. **Rastreable**: cada fila conserva `source_url` y `page_id` → se puede
   volver a la página original.

## Origen y confiabilidad

Toda la data viene de la prosa/HTML de **Gunter's Space Page**
(https://space.skyrocket.de), referencia curada por Gunter Dirk Krebs. No es
telemetría: las fechas son gruesas y las narraciones son de una persona.
Úsala como capa narrativa, no como dato primario (los datos primarios son
OMNI/DONKI/Space-Track).

> Contenido © Gunter Dirk Krebs 1996–2026, Gunter's Space Page
> (https://space.skyrocket.de). Usado bajo los términos de crawl del sitio
> (`robots.txt` permite crawlear; `noai`: no usar para entrenar modelos;
> RAG/summarización solo con atribución y link al original).

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from bs4 import BeautifulSoup

def _find_root(start):
    p = Path(start).resolve()
    while not (p / "data" / "gunter" / "meta" / "pages.parquet").exists() and p != p.parent:
        p = p.parent
    return p

ROOT = _find_root(Path.cwd())
DATA = ROOT / "data" / "gunter"

meta = pd.read_parquet(DATA / "meta/pages.parquet")
tables = pd.read_parquet(DATA / "tables.parquet")

doc_pages = meta[meta["url"].str.contains("/doc_sdat/", regex=False)].copy()
print(f"paginas doc_sdat: {len(doc_pages)} | registros de lanzamiento doc_sdat: "
      f"{(tables["source_url"].isin(doc_pages["url"])).sum()}")

paginas doc_sdat: 6710 | registros de lanzamiento doc_sdat: 32323


In [2]:
ENCODING = "iso-8859-1"

def parse_page_html(html: str):
    soup = BeautifulSoup(html, "html.parser")
    desc_el = soup.find(id="satdescription")
    description = ""
    if desc_el is not None:
        paras = [p.get_text(" ", strip=True) for p in desc_el.find_all("p")]
        description = "\n\n".join(paras)
    satdata = {}
    table = soup.find("table", id="satdata")
    if table is not None:
        for tr in table.find_all("tr"):
            th, td = tr.find("th"), tr.find("td")
            if th is not None and td is not None:
                key = th.get_text(" ", strip=True).rstrip(":")
                satdata[key] = td.get_text(" ", strip=True)
    return description, satdata

page_cache = {}
failed_parse = []
for i, row in doc_pages.iterrows():
    f = DATA / "pages" / f"{row['page_id']}.html"
    if not f.exists() or not f.stat().st_size:
        failed_parse.append(row["url"])
        continue
    page_cache[row["url"]] = parse_page_html(f.read_text(errors="replace", encoding=ENCODING))
    if (len(page_cache) in {1000, 3000, 5000, 6500}) or ((i + 1) == len(doc_pages)):
        print(f"parseadas {len(page_cache)}/{len(doc_pages)} paginas")

print("paginas sin parsear:", len(failed_parse))
print("ejemplo metadatos:", page_cache["https://space.skyrocket.de/doc_sdat/galaxy-15.htm"][1])

parseadas 1000/6710 paginas


parseadas 3000/6710 paginas


parseadas 5000/6710 paginas


parseadas 6106/6710 paginas


parseadas 6500/6710 paginas


paginas sin parsear: 0
ejemplo metadatos: {'Nation': 'USA', 'Type / Application': 'Communication', 'Operator': 'PanAmSat', 'Contractors': 'Orbital Sciences Corporation (OSC)', 'Equipment': '20-24 C-band transponders, WAAS payload', 'Configuration': 'Star-2 Bus', 'Propulsion': 'IHI BT-4', 'Power': '2 deployable solar arrays, batteries', 'Lifetime': '15 years', 'Mass': '2033\xa0kg (launch), 885\xa0kg (dry)', 'Orbit': 'GEO'}


In [3]:
pages_records = []
for url, (description, satdata) in page_cache.items():
    row = doc_pages.set_index("url").loc[url]
    pages_records.append({
        "source_url": url,
        "page_id": row["page_id"],
        "title": row["title"],
        "description_text": description,
        "nation": satdata.get("Nation"),
        "type_application": satdata.get("Type / Application"),
        "operator": satdata.get("Operator"),
        "contractors": satdata.get("Contractors"),
        "equipment": satdata.get("Equipment"),
        "configuration": satdata.get("Configuration"),
        "propulsion": satdata.get("Propulsion"),
        "power": satdata.get("Power"),
        "lifetime": satdata.get("Lifetime"),
        "mass": satdata.get("Mass"),
        "orbit": satdata.get("Orbit"),
    })
pages_df = pd.DataFrame(pages_records)
pages_df["mentions_years"] = pages_df["description_text"].apply(
    lambda s: sorted({int(y) for y in re.findall(r"\b(?:19|20)\d{2}\b", str(s))})
)
print(pages_df.shape)

(6710, 16)


In [4]:
def parse_launch_date(s):
    if not isinstance(s, str) or not s.strip() or s.strip() == "-":
        return pd.NaT
    try:
        return pd.to_datetime(s, format="%d.%m.%Y", errors="raise")
    except (ValueError, TypeError):
        return pd.to_datetime(s, errors="coerce", dayfirst=True)

registry = tables[tables["source_url"].isin(page_cache)].copy()
registry = registry.rename(columns={"Satellite": "satellite", "COSPAR": "cospar",
                                    "LS": "launch_site", "Launch Vehicle": "launch_vehicle",
                                    "Remarks": "remarks"})
registry["launch_date"] = registry["Date"].apply(parse_launch_date)
registry["launch_date_raw"] = registry["Date"]

tabla = registry.merge(pages_df, on="source_url", how="left")
tabla = tabla.drop(columns=["Date", ""])
print("filas objeto:", len(tabla))
print(tabla[["satellite", "cospar", "launch_date", "launch_vehicle", "orbit"]].head(3).to_string(index=False))

filas objeto: 32323
                            satellite   cospar launch_date        launch_vehicle                         orbit
Mars Telecommunications Orbiter (MTO)        -  2028-01-01                       Heliocentric, then Mars orbit
                         Slippers2sat 2025-292  2025-12-10 Lijian-1 (Kinetica-1)                              
               AIRSAT 11 (Zhongke 11)        -  2026-01-01                                                    


In [5]:
print("=== Tamaño / cobertura ===")
print("filas objeto:", len(tabla))
print("páginas únicas:", tabla["source_url"].nunique())
print("COSPAR no vacíos:", tabla["cospar"].ne("").sum(), "/", len(tabla))
print("con narrativa:", tabla["description_text"].fillna("").ne("").sum())
print()
print("=== % NaN por columna (top) ===")
for col, pct in tabla.isna().mean().sort_values(ascending=False).head(8).items():
    print(f"  {col:<28} {pct*100:5.1f}%")
print()
print("=== Ejemplo: Galaxy 15 ===")
g = tabla[(tabla["satellite"].str.contains("Galaxy 15", regex=False, na=False) & tabla["source_url"].str.contains("galaxy-15"))].head(1)
print(g[["satellite", "cospar", "launch_date", "operator", "orbit", "mentions_years"]].T)
print()
print("=== Ejemplo: prosa (razón narrada) de Galaxy 15 ===")
print(g["description_text"].iloc[0][-500:])

=== Tamaño / cobertura ===
filas objeto: 32323
páginas únicas: 5152
COSPAR no vacíos: 32323 / 32323
con narrativa: 32323

=== % NaN por columna (top) ===
  launch_date                    9.2%
  type_application               1.1%
  cospar                         0.0%
  launch_site                    0.0%
  launch_vehicle                 0.0%
  remarks                        0.0%
  satellite                      0.0%
  source_url                     0.0%

=== Ejemplo: Galaxy 15 ===
                                     2511
satellite       Galaxy 15 (ex Galaxy 1RR)
cospar                          2005-041A
launch_date           2005-10-13 00:00:00
operator                         PanAmSat
orbit                                 GEO
mentions_years   [2001, 2003, 2010, 2022]

=== Ejemplo: prosa (razón narrada) de Galaxy 15 ===
er 2010 leading to a reset of the systems. After that, control was regained over the satellite.

In August 2022 Intelsat again lost the ability to command its Galaxy 1

In [6]:
out_parquet = DATA / "gunter_tabular.parquet"
out_csv = DATA / "gunter_tabular.csv"
tabla.to_parquet(out_parquet, index=False)
tabla.to_csv(out_csv, index=False)
print("escrito:", out_parquet, "->", out_parquet.stat().st_size // 1024, "KiB")
print("escrito:", out_csv, "->", out_csv.stat().st_size // 1024, "KiB")

escrito: /home/pxtron/Documents/cme-sentinel/data/gunter/gunter_tabular.parquet -> 8045 KiB
escrito: /home/pxtron/Documents/cme-sentinel/data/gunter/gunter_tabular.csv -> 67830 KiB


## Diccionario de datos y exploración

Esta sección documenta qué significa cada columna de la tabla tabular
(`gunter_tabular.parquet`) y muestra casos concretos. Convenciones del dataset:

- **Fila** = un registro `#satlist` (objeto/lanzamiento) del sitio. Una página
  `doc_sdat` (p. ej. una constelación) puede alimentar **muchas filas**, una por
  objeto. `source_url` + `page_id` permiten volver a la página original.
- **Vacío vs. null**: el sitio no reporta un dato con `""` (o no lo escribe), no
  con `NaN`. `NaN` aparece solo cuando el parseo de ese campo no encontró valor.
- **`cospar == "-"`** = aún sin designación COSPAR (objeto planeado o fallido).
- **Fechas gruesas**: `launch_date` es `datetime64[UTC]`; cuando el sitio solo
  indica el año ("2028"), se guarda como el 1 de enero de ese año. La columna
  `launch_date_raw` conserva el texto original (incluye incertidumbres tipo
  "202x"), así que es la fuente de verdad para fechas parciales.
- **`orbit` es texto libre** ("550 km × 550 km, 53° (typical)"), no numérico:
  no se puede unir directamente con `gp_history` sin normalizarlo.
- **`mentions_years`** = lista de años (regex `19xx`/`20xx`) citados en la
  narrativa. Señala fechas mencionadas en la historia, no eventos estructurados.

### Identidad y trazabilidad

| Columna | Significado |
|---|---|
| `satellite` | Nombre del objeto según el registro `#satlist` (ej. "Starlink v1.5 G2-1-1 (Starlink 3096)"). Una fila por objeto/lanzamiento. |
| `cospar` | Designación COSPAR/INTLDES (ej. "2021-082B"). `"-"` = sin designación (planeado o no lanzado). |
| `title` | Título de la página `doc_sdat` en el sitio. |
| `page_id` | Hash del HTML crudo guardado en `data/gunter/pages/<page_id>.html`. |
| `source_url` | URL canónica de la página; llave de unión con `tables.parquet` e `incidents.parquet`. |

### Lanzamiento

| Columna | Significado |
|---|---|
| `launch_date` | Fecha de lanzamiento parseada a `datetime64[µs]` UTC. Gruesa: solo año → `01-01` de ese año. `NaT` si el sitio no indica fecha. |
| `launch_date_raw` | Texto original de la fecha tal como aparece ("14.09.2021", "2028", "202x"). |
| `launch_site` | Código del puerto espacial (ej. "Va SLC-4E", "CC SLC-40"). |
| `launch_vehicle` | Vehículo lanzador (ej. "Falcon-9 v1.2 (Block 5)"). |
| `remarks` | Notas del registro de lanzamiento (con qué otros objetos voló). |

### Narrativa

| Columna | Significado |
|---|---|
| `description_text` | Prosa del campo `#satdescription`: historial del objeto, fallas, anomalías, destino. La capa narrativa (fechas gruesas, curada por una persona, no telemetría). |
| `mentions_years` | Años citados en la narrativa (extraídos con regex). |

### Ficha técnica (`#satdata`)

| Columna | Significado |
|---|---|
| `nation` | Nación base del operador (ej. "USA", "USSR", "China"). |
| `type_application` | Tipo/aplicación (ej. "Communication", "Earth observation", "Technology"). |
| `operator` | Operador (ej. "SpaceX", "NASA"). |
| `contractors` | Contratistas / fabricante del bus. |
| `equipment` | Carga útil / equipamiento (ej. transponders, SAR, enlaces ópticos). |
| `configuration` | Arquitectura del bus (ej. "Star-2 Bus", "CubeSat (1U)"). |
| `propulsion` | Sistema de propulsión (o "None"). |
| `power` | Fuente de alimentación (paneles solares, baterías...). |
| `lifetime` | Vida útil nominal (ej. "15 years"). Muy frecuentemente vacío. |
| `mass` | Masa del objeto (a veces con masa de otros de la serie). |
| `orbit` | Órbita descrita en texto libre (ej. "GEO", "550 km × 550 km, 53° (typical)"). |

In [7]:
print("=== Cobertura por columna ===\n")

rows = []
for c in tabla.columns:
    s = tabla[c]
    na = s.isna()
    nonempty = s.notna()
    if "str" in str(s.dtype):
        nonempty = s.notna() & s.astype(str).ne("")
    try:
        nunique = s.nunique()
    except TypeError:
        nunique = "(lista)"
    ex = s.dropna().iloc[0] if s.notna().any() else None
    sample = " ".join(str(ex).split())[:70]
    rows.append([c, str(s.dtype), f"{nonempty.mean()*100:.1f}%",
                f"{na.mean()*100:.1f}%", nunique, sample])
cov = pd.DataFrame(rows, columns=["columna", "dtype", "% no vacío", "% NaN", "únicos", "ejemplo"])
print(cov.to_string(index=False))

=== Cobertura por columna ===

         columna          dtype % no vacío % NaN  únicos                                                                ejemplo
       satellite            str     100.0%  0.0%   32056                                  Mars Telecommunications Orbiter (MTO)
          cospar            str     100.0%  0.0%   27743                                                                      -
     launch_site            str      93.5%  0.0%     293                                                                     CC
  launch_vehicle            str      93.6%  0.0%     736                                                                       
         remarks            str      81.2%  0.0%    6537                                              with UHF relay satellites
      source_url            str     100.0%  0.0%    5152                         https://space.skyrocket.de/doc_sdat/mto_bo.htm
     launch_date datetime64[us]      90.8%  9.2%    6410                 

### El dataframe tabular

Vista directa de las primeras 10 filas de la tabla final (23 columnas): una fila por objeto/lanzamiento, con la ficha `#satdata`, la narrativa y la trazabilidad a la página original.

In [8]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 40)
print(tabla.head(10).to_string(index=False))

                            satellite   cospar  launch_site          launch_vehicle                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   remarks                                           source_url launch_date launch_date_raw  page_id                                    title                                                                                                                                                                                                                                                        

In [9]:
print("=== Valores más comunes por columna clave ===\n")

for c, label in [("nation", "País nacional (nation)"),
                 ("type_application", "Tipo / aplicación"),
                 ("orbit", "Órbita (texto libre)"),
                 ("launch_site", "Sitio de lanzamiento"),
                 ("launch_vehicle", "Vehículo lanzador")]:
    print(f"--- {label} (top 6) ---")
    print(tabla[c].fillna("(null)").replace("", "(vacío)").value_counts().head(6).to_string())
    print()

print("=== Años citados en la narrativa (mentions_years, últimos 14) ===\n")
years = pd.Series([int(y) for ys in tabla["mentions_years"].dropna() for y in ys])
print(years.value_counts().sort_index().tail(14).to_string())
print("\n(total menciones de año:", len(years), ")")

=== Valores más comunes por columna clave ===

--- País nacional (nation) (top 6) ---
nation
USA                     20845
USSR                     2274
China                    2080
USSR / Russia            1082
Russia                    974
UK (Channel Islands)      689

--- Tipo / aplicación (top 6) ---
type_application
Communication                               15944
Technology                                   2501
Earth observation                            1207
Experimental Communication                    941
Reconnaissance, photo (film return type)      893
Navigation                                    738

--- Órbita (texto libre) (top 6) ---
orbit
550 km × 550 km, 53° (typical)                                      12912
(vacío)                                                              8482
GEO                                                                  1603
410 km × 410 km, 51.66° (#1, #1b); 605 km × 620 km, 97.99° (#1c)      745
1200 km × 1200 km, ?°              

### Casos de ejemplo

Tres filas representativas, alineadas con la investigación CME → satélite:

1. **Galaxy 15** (`galaxy-15.htm`, COSPAR `2005-041A`) — anomalía por un
   **evento de clima espacial**: en agosto 2022 Intelsat perdió el control del
   satélite ("an anomaly caused by a space weather event"), perdió los enlaces
   de mando y derivó de su estación. Es la **vía electrónica** narrada como
   prosa: ideal para validación puntual, no para event study.
2. **Starlink v1.5 G2-1-1** (Starlink 3096, COSPAR `2021-082B`) — fila típica
   de un objeto LEO comercial de una tanda de 51. Muestra cómo se ve la **vía
   orbital** a nivel de catálogo (LEO ~550 km, masa ~300 kg) y que `orbit` es
   texto libre agrupable por regex por familia.
3. **Mars Telecommunications Orbiter** — objeto **no lanzado**: `cospar == "-"`
   y fecha gruesa `2028` → `2028-01-01`. Ilustra los sentinels del esquema.

Los objetos no lanzados (`cospar == "-"`, 3467 filas ≈ 11%) se deben filtrar
antes de cualquier cruce con `gp_history`.

In [10]:
def _show(row, cols, title):
    print(f"{title}\n")
    for c in cols:
        v = row[c]
        print(f"  {c:<18} {v}")
    print()

cols = ["satellite", "cospar", "launch_date", "launch_date_raw", "launch_site",
        "launch_vehicle", "nation", "type_application", "operator", "contractors",
        "equipment", "configuration", "propulsion", "power", "lifetime", "mass",
        "orbit", "mentions_years"]

g = tabla[tabla["source_url"].str.contains("galaxy-15.htm", regex=False)].head(1)
_show(g.iloc[0], cols, "=== Caso 1 — Galaxy 15 (anomalía por clima espacial narrada) ===")
d = g["description_text"].iloc[0]
i = d.lower().find("space weather event")
print("  Prosa (fragmento del evento de 2022):\n")
print("  «" + " ".join(d[max(0, i - 120): i + 200].split()) + "»\n\n")

sl = tabla[tabla["satellite"].astype(str).str.contains("Starlink 3096", regex=False)].head(1)
_show(sl.iloc[0], cols, "=== Caso 2 — Starlink v1.5 de la tanda G2-1 (objeto LEO masivo) ===")

mto = tabla[tabla["satellite"].astype(str).str.contains("Mars Telecommunications Orbiter", regex=False)].head(1)
_show(mto.iloc[0], cols, "=== Caso 3 — Objeto planeado no lanzado (sentinels del esquema) ===")

=== Caso 1 — Galaxy 15 (anomalía por clima espacial narrada) ===

  satellite          Galaxy 15 (ex Galaxy 1RR)
  cospar             2005-041A
  launch_date        2005-10-13 00:00:00
  launch_date_raw    13.10.2005
  launch_site        Ko ELA-3
  launch_vehicle     Ariane-5GS
  nation             USA
  type_application   Communication
  operator           PanAmSat
  contractors        Orbital Sciences Corporation (OSC)
  equipment          20-24 C-band transponders, WAAS payload
  configuration      Star-2 Bus
  propulsion         IHI BT-4
  power              2 deployable solar arrays, batteries
  lifetime           15 years
  mass               2033 kg (launch), 885 kg (dry)
  orbit              GEO
  mentions_years     [2001, 2003, 2010, 2022]

  Prosa (fragmento del evento de 2022):

  «ellite. In August 2022 Intelsat again lost the ability to command its Galaxy 15 satellite after an anomaly caused by a space weather event. The anomaly caused the loss of commanding links, which l

In [11]:
print("=== Minería de narrativa: keywords de clima espacial / órbita ===\n")

narr = tabla["description_text"].fillna("")
for kw in ["solar storm", "space weather", "geomagnetic", "radiation",
           "reenter", "re-entry", "decay", "anomal", "particle"]:
    n = narr.str.lower().str.count(kw).sum()
    print(f"  {kw:<14} {n:>6} menciones")

print("\n=== Ejemplos reales de reentry narrado (primeros 3) ===\n")
hits = tabla[narr.str.lower().str.contains("reenter", regex=False)]
for _, r in hits.head(3).iterrows():
    t = r["description_text"].lower()
    i = t.find("reenter")
    seg = " ".join(r["description_text"][max(0, i - 100): i + 130].split())
    print(f"  • {r["satellite"]}  ({r["source_url"].rsplit("/", 1)[-1]})")
    print(f"    …{seg}…")
    print()
print("\n(La narrativa exige revisión manual: mención ≠ evento causal.)")

=== Minería de narrativa: keywords de clima espacial / órbita ===

  solar storm         2 menciones


  space weather     207 menciones
  geomagnetic       110 menciones


  radiation        1770 menciones


  reenter           481 menciones
  re-entry          288 menciones
  decay             293 menciones


  anomal            161 menciones


  particle         1218 menciones

=== Ejemplos reales de reentry narrado (primeros 3) ===



  • HASTE  (curveball.htm)
    …ion implied by pre-launch hazard areas in the western Atlantic. Both rocket stages were expected to reenter the atmosphere within approximately two days of launch. The payload itself was not cataloged. The most likely explanation…

  • Yodaka (AE1b)  (ae1b.htm)
    …e using the J-SSOD small-satellite deployer. The satellite remained in orbit for several months and reentered the atmosphere on 28 February 2025.…

  • ATENEA  (atenea.htm)
    …is 2 mission, which will conduct a crewed test flight around the Moon. ATENEA was expected to have reentered at 1st perigee since it had no propulsion to raise its orbit.…


(La narrativa exige revisión manual: mención ≠ evento causal.)
